In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import pandas as pd
import polars as pl
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time

import sys


from pyS3M import IOFunctions

IO = IOFunctions.IO_Functions()

from pyS3M import PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

from pyS3M import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from pyS3M import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from pyS3M import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from pyS3M import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from pyS3M import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from pyS3M import SpotDetectionFunctions

SD_F = SpotDetectionFunctions.SpotDetection_Functions()

from pyS3M import SR_Functions

SRes_F = SR_Functions.SuperRes_Functions()

from pyS3M import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

from pyS3M import SM_extractionfunctions

SM_E = SM_extractionfunctions.extract_SMs()

from pyS3M.DriftCorrectionFunctions import (
    Drift_Correction_Functions,
    DriftMethod,
    DriftParameters,
    DriftResult,
)

import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

/tmp/ipykernel_816180/1288529626.py:22: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter(dark_background=False)


In [2]:
data_folder = "../../Camera_Calibrations/Ximea_Camera/"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

In [3]:
objective_T = S_F.getobjectiveefficiency(wavelength)

In [4]:
wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])

In [ ]:
folders = '/scratch/sycamore-asap/2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments'
folder_in_folder = os.listdir(folders)

min_cluster_size = 10
chi_val = 2
max_localisation_error = 1.0
min_photons = 100
max_photons = 1e6
max_distance = 0.5

for folder in folder_in_folder:
    print('Analysing {}'.format(folder))
    example_folder = os.path.join(folders, folder)
    localisation_files = H_F.file_search(example_folder, ".h5", "Pos")
    localisation_files = np.sort([x for x in localisation_files if 'database' not in x])
    old_databases = H_F.file_search(example_folder, ".h5", "database")
    old_files = np.sort([x for x in old_databases if 'old' not in x])
    if len(old_files) > 0:
        for file in old_files:
            os.rename(file, file.split('.h5')[0]+'_old.h5')
    single_molecule_database, single_frame_database = SM_E.extract_single_molecules_batch(
        localisation_files,
        clustering_method="HDBSCAN",
        chi_val=chi_val,
        min_cluster_size=min_cluster_size,
        max_localisation_error=max_localisation_error,
        min_photons=min_photons,
        max_photons=max_photons,
    )
    single_molecule_database.to_hdf(os.path.join(example_folder, 'Single_molecule_database.h5'), key="data", append=False, format="table")
    single_frame_database.to_hdf(os.path.join(example_folder, 'Single_frame_database.h5'), key="data", append=False, format="table")